# Adquisición y Cruce de Datos
**Proyecto:** Predicción de Satisfacción en Aerolíneas según Servicio y Perfil Sociodemográfico.
**Objetivo de este notebook:** Leer las fuentes originales, realizar el cruce estadístico por edad y género, y consolidar el diccionario de datos.

In [26]:
import pandas as pd
import numpy as np

# Configuración para ver todas las columnas si el dataset es muy ancho
pd.set_option('display.max_columns', None)

Se cargan los datasets desde la carpeta `/data`. 
* **Fuente 1:** Pasajeros de Aerolínea (Kaggle)
* **Fuente 2:** Censo de Ingresos Adultos (Kaggle)

In [27]:
# Leer Fuente 1: Pasajeros
pasajeros = pd.read_csv('../data/airline_passenger_satisfaction.csv')
print(f"Dimensiones de Pasajeros: {pasajeros.shape}")
display(pasajeros.head(3))

# Leer Fuente 2: Censo
censo = pd.read_csv('../data/adult_census_income.csv')
print(f"\nDimensiones de Censo: {censo.shape}")
display(censo.head(3))

Dimensiones de Pasajeros: (103904, 25)


,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,70172,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,3,1,5,3,5,5,4,3,4,4,5,5,25,18.0,neutral or dissatisfied
1,1,5047,Male,disloyal Customer,25,Business travel,Business,235,3,2,3,3,1,3,1,1,1,5,3,1,4,1,1,6.0,neutral or dissatisfied
2,2,110028,Female,Loyal Customer,26,Business travel,Business,1142,2,2,2,2,5,5,5,5,4,3,4,4,4,5,0,0.0,satisfied



Dimensiones de Censo: (32561, 15)


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K


Los nombres de las columnas varían entre los datasets (ej. `Age` vs `age`, `Gender` vs `sex`). se van a estandarizar en el censo para que coincidan con los de pasajeros. 
Además, se creara una variable numérica `ingreso_alto` en el censo para poder calcular promedios.

In [28]:
# Estandarizar nombres de columnas en el Censo
censo = censo.rename(columns={'age': 'Age', 'sex': 'Gender'})

# El censo americano usa "Male" y "Female", pero a veces con espacios extra. 
# Se asegura el mismo formato de la fuente de pasajeros.
censo['Gender'] = censo['Gender'].str.strip().str.capitalize()
pasajeros['Gender'] = pasajeros['Gender'].str.strip().str.capitalize()

# 2.2 Convertir el ingreso del censo a numérico (1 si gana >50K, 0 si no)
# Ajusta '>50K' según cómo venga escrito en el CSV (a veces trae un punto final '>50K.')
censo['ingreso_alto'] = np.where(censo['income'].str.contains('>50K'), 1, 0)

**Caso de cruce por aproximación:** Dado que no se tienen a las mismas personas físicas, calcularemos un estimado del poder adquisitivo (probabilidad de ingresos altos) agrupando por **Edad y Género** en el censo, y lo inyectaremos a los pasajeros.

In [29]:
# Calcular el promedio estadístico de ingresos por perfil
ingreso_promedio = censo.groupby(["Age", "Gender"])["ingreso_alto"].mean().reset_index()
ingreso_promedio = ingreso_promedio.rename(columns={'ingreso_alto': 'probabilidad_altos_ingresos'})

# Verificación antes del cruce
filas_antes = pasajeros.shape[0]
print(f"Filas en dataset principal ANTES del cruce: {filas_antes}")

#se hace cruce
pasajeros_cruzado = pd.merge(pasajeros, ingreso_promedio, on=["Age", "Gender"], how="left")

# Verificación después del cruce
filas_despues = pasajeros_cruzado.shape[0]
print(f"Filas en dataset principal DESPUÉS del cruce: {filas_despues}")

# Comprobación de integridad
if filas_antes == filas_despues:
    print("¡Éxito! El cruce no duplicó ni perdió pasajeros.")
else:
    print("Advertencia: Las filas cambiaron. Revisar duplicados en las llaves.")

display(pasajeros_cruzado[['id', 'Age', 'Gender', 'probabilidad_altos_ingresos', 'satisfaction']].head())

Filas en dataset principal ANTES del cruce: 103904
Filas en dataset principal DESPUÉS del cruce: 103904
¡Éxito! El cruce no duplicó ni perdió pasajeros.


,id,Age,Gender,probabilidad_altos_ingresos,satisfaction
0,70172,13,Male,NaN,neutral or dissatisfied
1,5047,25,Male,0.071970,neutral or dissatisfied
2,110028,26,Female,0.048276,satisfied
3,24026,25,Female,0.047923,neutral or dissatisfied
4,119299,61,Male,0.407240,satisfied


A continuación, el diccionario de las columnas principales del dataset ya consolidado (`pasajeros_cruzado`), el cual responde directamente a nuestro objetivo de negocio.

| Columna | Tipo de Dato (Pandas) | Significado | Fuente Origen |
| :--- | :--- | :--- | :--- |
| `id` | int64 | Identificador único del pasajero encuestado. | Kaggle: Airline Passengers |
| `Gender` | object | Género del pasajero (Male / Female). | Kaggle: Airline Passengers |
| `Age` | int64 | Edad real del pasajero en años. | Kaggle: Airline Passengers |
| `Customer Type` | object | Nivel de lealtad (Loyal customer / disloyal customer). | Kaggle: Airline Passengers |
| `Type of Travel` | object | Motivo del viaje (Personal Travel / Business Travel). | Kaggle: Airline Passengers |
| `Class` | object | Cabina en la que viajó (Business, Eco, Eco Plus). | Kaggle: Airline Passengers |
| `Inflight wifi service` | int64 | Calificación del WiFi (0 a 5). | Kaggle: Airline Passengers |
| `Leg room service` | int64 | Calificación del espacio para piernas (0 a 5). | Kaggle: Airline Passengers |
| `probabilidad_altos_ingresos` | float64 | Valor inferido (0.0 a 1.0). Indica la probabilidad de que el pasajero gane +50K al año basado en su edad y género. | Kaggle: Adult Census |
| `Satisfaction` | object | Variable Objetivo. Si quedó satisfecho o insatisfecho. | Kaggle: Airline Passengers |